# Session 9: Protocols for Network Management: NTP, DHCP, DNS, TFTP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-09-network-management-protocols.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-09_dhcp_pool.py`

In [1]:
"""Session 9 - A DHCP server's view: pool, exclusions, DORA and leases.

Mirrors what "ip dhcp pool" on a Cisco router does: hand out the lowest free
address that is not excluded, remember the lease against the client MAC,
renew at T1 (50% of the lease), and release addresses that expire.
Run:  python session-09_dhcp_pool.py
"""
import ipaddress


class DhcpServer:
    def __init__(self, network, gateway, dns, lease_s, excluded):
        self.net = ipaddress.ip_network(network)
        self.gateway, self.dns, self.lease_s = gateway, dns, lease_s
        self.excluded = set()
        for lo, hi in excluded:
            a, b = int(ipaddress.ip_address(lo)), int(ipaddress.ip_address(hi))
            self.excluded |= {ipaddress.ip_address(x) for x in range(a, b + 1)}
        self.bindings = {}            # mac -> (ip, expiry time)

    def free(self):
        used = {ip for ip, _ in self.bindings.values()}
        return [h for h in self.net.hosts() if h not in self.excluded and h not in used]

    def dora(self, mac, now):
        print(f"  DISCOVER  from {mac} (broadcast, src 0.0.0.0 -> 255.255.255.255, UDP 68->67)")
        if mac in self.bindings:
            ip = self.bindings[mac][0]
        else:
            pool = self.free()
            if not pool:
                print("  (no OFFER: pool exhausted - client falls back to APIPA 169.254.x.x)")
                return None
            ip = pool[0]
        print(f"  OFFER     {ip}/{self.net.prefixlen} gw {self.gateway} dns {self.dns} lease {self.lease_s}s")
        print(f"  REQUEST   {mac} asks for {ip} (broadcast, so other servers withdraw their offers)")
        self.bindings[mac] = (ip, now + self.lease_s)
        print(f"  ACK       {ip} bound to {mac}, renew (T1) at t={now + self.lease_s // 2}s")
        return ip

    def assign(self, mac, now):
        """Same result as DORA, without printing each message."""
        pool = self.free()
        if mac not in self.bindings and not pool:
            return None
        ip = self.bindings[mac][0] if mac in self.bindings else pool[0]
        self.bindings[mac] = (ip, now + self.lease_s)
        return ip

    def expire(self, now):
        gone = [m for m, (_, exp) in self.bindings.items() if exp <= now]
        for m in gone:
            print(f"  lease of {self.bindings[m][0]} for {m} expired and returned to the pool")
            del self.bindings[m]

    def show_bindings(self):
        print("  IP address       Client-ID/Hardware address   Lease expiration (t)")
        for mac, (ip, exp) in sorted(self.bindings.items(), key=lambda x: x[1][0]):
            print(f"  {str(ip):16} {mac:28} {exp}s")


if __name__ == "__main__":
    srv = DhcpServer("192.168.10.0/28", "192.168.10.1", "192.168.10.5", lease_s=3600,
                     excluded=[("192.168.10.1", "192.168.10.5")])
    print(f"Pool 192.168.10.0/28: {len(list(srv.net.hosts()))} hosts, 5 excluded -> {len(srv.free())} leasable\n")
    clients = [f"aaaa.bbbb.{i:04x}" for i in range(1, 12)]
    for t, mac in enumerate(clients[:3]):
        print(f"t={t}s"); srv.dora(mac, t); print()
    print("Bindings (like 'show ip dhcp binding'):"); srv.show_bindings()
    print("\nEight more clients arrive at t=10s")
    for m in clients[3:]:
        ip = srv.assign(m, 10)
        print(f"  {m} -> {ip if ip else 'NO ADDRESS (pool exhausted, client uses APIPA 169.254.x.x)'}")
    print("\nAt t=3601s the first leases expire:")
    srv.expire(3601)
    print(f"  free addresses now: {', '.join(map(str, srv.free()))}")

Pool 192.168.10.0/28: 14 hosts, 5 excluded -> 9 leasable

t=0s
  DISCOVER  from aaaa.bbbb.0001 (broadcast, src 0.0.0.0 -> 255.255.255.255, UDP 68->67)
  OFFER     192.168.10.6/28 gw 192.168.10.1 dns 192.168.10.5 lease 3600s
  REQUEST   aaaa.bbbb.0001 asks for 192.168.10.6 (broadcast, so other servers withdraw their offers)
  ACK       192.168.10.6 bound to aaaa.bbbb.0001, renew (T1) at t=1800s

t=1s
  DISCOVER  from aaaa.bbbb.0002 (broadcast, src 0.0.0.0 -> 255.255.255.255, UDP 68->67)
  OFFER     192.168.10.7/28 gw 192.168.10.1 dns 192.168.10.5 lease 3600s
  REQUEST   aaaa.bbbb.0002 asks for 192.168.10.7 (broadcast, so other servers withdraw their offers)
  ACK       192.168.10.7 bound to aaaa.bbbb.0002, renew (T1) at t=1801s

t=2s
  DISCOVER  from aaaa.bbbb.0003 (broadcast, src 0.0.0.0 -> 255.255.255.255, UDP 68->67)
  OFFER     192.168.10.8/28 gw 192.168.10.1 dns 192.168.10.5 lease 3600s
  REQUEST   aaaa.bbbb.0003 asks for 192.168.10.8 (broadcast, so other servers withdraw their off

## `session-09_dns_ntp.py`

In [2]:
"""Session 9 - Look inside DNS, NTP and TFTP messages (built and parsed offline).

1. Build a real DNS query for an A record and parse a sample answer.
2. Compute NTP clock offset and delay from the four timestamps.
3. Build TFTP read-request and data packets (how a router backs up its config).
Nothing is sent on the network.
Run:  python session-09_dns_ntp.py
"""
import struct


def dns_query(name, qtype=1, ident=0x1A2B):
    header = struct.pack("!HHHHHH", ident, 0x0100, 1, 0, 0, 0)   # RD flag set, 1 question
    qname = b"".join(len(p).to_bytes(1, "big") + p.encode() for p in name.split(".")) + b"\0"
    return header + qname + struct.pack("!HH", qtype, 1)          # type A, class IN


def dns_answer_for(query, ip, ttl=300):
    ident = struct.unpack("!H", query[:2])[0]
    header = struct.pack("!HHHHHH", ident, 0x8180, 1, 1, 0, 0)    # response, RD+RA, no error
    answer = b"\xc0\x0c" + struct.pack("!HHIH", 1, 1, ttl, 4) + bytes(map(int, ip.split(".")))
    return header + query[12:] + answer                             # c00c = pointer to the name


def parse_answer(msg):
    ident, flags, qd, an, _, _ = struct.unpack("!HHHHHH", msg[:12])
    i, labels = 12, []
    while msg[i]:
        labels.append(msg[i + 1:i + 1 + msg[i]].decode()); i += msg[i] + 1
    i += 5                                                          # zero byte + qtype + qclass
    _, rtype, _, ttl, rdlen = struct.unpack("!HHHIH", msg[i:i + 12])
    ip = ".".join(map(str, msg[i + 12:i + 12 + rdlen]))
    rcode = flags & 0xF
    return {"id": hex(ident), "name": ".".join(labels), "answers": an, "type": rtype, "ttl": ttl,
            "ip": ip, "rcode": rcode}


def ntp_offset(t1, t2, t3, t4):
    """t1 client send, t2 server receive, t3 server send, t4 client receive."""
    return ((t2 - t1) + (t3 - t4)) / 2, (t4 - t1) - (t3 - t2)


def tftp_request(opcode, filename, mode="octet"):
    """opcode 1 = RRQ (read/download), 2 = WRQ (write/upload)."""
    return struct.pack("!H", opcode) + filename.encode() + b"\0" + mode.encode() + b"\0"


def tftp_data(block, data):
    return struct.pack("!HH", 3, block) + data


if __name__ == "__main__":
    q = dns_query("portal.lab.local")
    print(f"DNS query ({len(q)} bytes, UDP 53): {q.hex(' ')}")
    a = dns_answer_for(q, "10.20.30.40")
    print(f"DNS answer ({len(a)} bytes) parsed: {parse_answer(a)}")

    print("\nNTP: client clock is behind the server")
    t1, t2, t3, t4 = 1000.000, 1002.530, 1002.531, 1000.061
    off, delay = ntp_offset(t1, t2, t3, t4)
    print(f"  t1={t1} t2={t2} t3={t3} t4={t4}")
    print(f"  offset = {off:+.4f} s (client must move forward), round-trip delay = {delay:.4f} s")
    for stratum, src in [(0, "atomic clock / GPS"), (1, "server attached to stratum 0"),
                         (2, "core router syncing from stratum 1"), (3, "access switches syncing from the core")]:
        print(f"  stratum {stratum}: {src}")

    print("\nTFTP (UDP 69): router backs up its config to a TFTP server")
    wrq = tftp_request(2, "R1-confg")
    print(f"  WRQ  opcode 2 (upload/backup): {wrq}")
    print(f"  RRQ  opcode 1 (download/restore) would be: {tftp_request(1, 'R1-confg')}")
    config = b"hostname R1\n!\ninterface g0/0\n ip address 192.168.10.1 255.255.255.0\n"
    blocks = [config[i:i + 512] for i in range(0, len(config), 512)]
    for n, b in enumerate(blocks, 1):
        print(f"  DATA opcode 3 block {n}: {len(b)} bytes (< 512 means last block) -> ACK {n}")
    print("  No login and no encryption: use TFTP only on a trusted management network.")

DNS query (34 bytes, UDP 53): 1a 2b 01 00 00 01 00 00 00 00 00 00 06 70 6f 72 74 61 6c 03 6c 61 62 05 6c 6f 63 61 6c 00 00 01 00 01
DNS answer (50 bytes) parsed: {'id': '0x1a2b', 'name': 'portal.lab.local', 'answers': 1, 'type': 1, 'ttl': 300, 'ip': '10.20.30.40', 'rcode': 0}

NTP: client clock is behind the server
  t1=1000.0 t2=1002.53 t3=1002.531 t4=1000.061
  offset = +2.5000 s (client must move forward), round-trip delay = 0.0600 s
  stratum 0: atomic clock / GPS
  stratum 1: server attached to stratum 0
  stratum 2: core router syncing from stratum 1
  stratum 3: access switches syncing from the core

TFTP (UDP 69): router backs up its config to a TFTP server
  WRQ  opcode 2 (upload/backup): b'\x00\x02R1-confg\x00octet\x00'
  RRQ  opcode 1 (download/restore) would be: b'\x00\x01R1-confg\x00octet\x00'
  DATA opcode 3 block 1: 68 bytes (< 512 means last block) -> ACK 1
  No login and no encryption: use TFTP only on a trusted management network.
